# Australia 2026 — GridRival Race Analysis

This notebook walks through the `gr_analytics` package using the actual results from the 2026 Australian Grand Prix.

**Topics covered:**
1. Scoring the race
2. Inspecting driver and constructor results
3. Finding the optimal lineup (points and salary)
4. Scoring a specific team selection

## 1. Race Scenario

The scenario captures each driver's qualifying position, whether they completed qualifying, their finishing position, and what fraction of the race they completed.

In [ ]:
import pandas as pd
from gr_analytics import score_event, score_my_team, optimal_lineup

# Australia 2026 race results
australia = pd.DataFrame({
    "driver_abbr":          ["VER", "RUS", "NOR", "PIA", "ANT", "LEC", "ALO", "HAM",
                              "HAD", "GAS", "STR", "SAI", "LAW", "ALB", "HUL", "BOR",
                              "BEA", "OCO", "BOT", "PER", "COL", "LIN"],
    "qualifying_position":  [20, 1, 6, 5, 2, 4, 17, 7,
                              3, 14, 22, 21, 8, 15, 11, 10,
                              12, 13, 19, 18, 16, 9],
    "completed_qualifying": [0, 1, 1, 1, 1, 1, 1, 1,
                              1, 1, 0, 0, 1, 1, 1, 1,
                              1, 1, 1, 1, 1, 1],
    "finishing_position":   [6, 1, 5, 21, 2, 3, 18, 4,
                              20, 10, 17, 15, 13, 12, 22, 9,
                              7, 11, 19, 16, 14, 8],
    "completed_pct":        [1, 1, 1, "DNS", 1, 1, 0.3, 1,
                              0.1, 1, 0.76, 1, 1, 1, "DNS", 1,
                              1, 1, 0.3, 1, 1, 1],
})

australia

,driver_abbr,qualifying_position,completed_qualifying,finishing_position,completed_pct
0,VER,20,0,6,1
1,RUS,1,1,1,1
2,NOR,6,1,5,1
3,PIA,5,1,21,DNS
4,ANT,2,1,2,1
5,LEC,4,1,3,1
6,ALO,17,1,18,0.3
7,HAM,7,1,4,1
8,HAD,3,1,20,0.1
9,GAS,14,1,10,1


## 2. Score the Race

`score_event` merges the scenario with pre-season driver data (round 0) and computes all fantasy points and post-race salaries.

In [2]:
scored = score_event(australia, round=0)

### Driver results

In [ ]:
driver_cols = [
    "driver_abbr", "driver_name",
    "pts_qualifying", "pts_race", "pts_overtake",
    "pts_improvement", "pts_completion", "pts_teammate",
    "points_earned", "starting_salary", "salary_after_event", "salary_change",
]

drivers = (
    scored[scored["type"] == "driver"][driver_cols]
    .sort_values("points_earned", ascending=False)
    .reset_index(drop=True)
)
drivers

,driver_abbr,driver_name,pts_qualifying,pts_race,pts_overtake,pts_improvement,pts_completion,pts_teammate,points_earned,starting_salary,salary_after_event,salary_change
0,BEA,O. Bearman,28,82,15.0,30.0,12.0,5.0,172.0,9.2,11.2,2.0
1,RUS,G. Russell,50,100,0.0,0.0,12.0,2.0,164.0,28.7,29.6,0.9
2,LIN,A. Lindblad,34,79,3.0,30.0,12.0,5.0,163.0,4.6,6.6,2.0
3,ANT,K. Antonelli,48,97,0.0,4.0,12.0,0.0,161.0,24.8,25.9,1.1
4,LEC,C. Leclerc,44,94,3.0,4.0,12.0,2.0,159.0,23.5,24.5,1.0
5,HAM,L. Hamilton,38,91,9.0,6.0,12.0,0.0,156.0,20.9,22.1,1.2
6,NOR,L. Norris,40,88,3.0,0.0,12.0,12.0,155.0,27.4,26.7,-0.7
7,BOR,G. Bortoleto,32,76,3.0,16.0,12.0,12.0,151.0,10.5,12.5,2.0
8,VER,M. Verstappen,0,85,42.0,0.0,12.0,12.0,151.0,30.0,28.2,-1.8
9,OCO,E. Ocon,26,70,6.0,16.0,12.0,0.0,130.0,7.9,9.9,2.0


### Constructor results

In [ ]:
constructor_cols = [
    "driver_name", "pts_qualifying", "pts_race",
    "points_earned", "starting_salary", "salary_after_event", "salary_change",
]

constructors = (
    scored[scored["type"] == "team"][constructor_cols]
    .sort_values("points_earned", ascending=False)
    .reset_index(drop=True)
)
constructors

,driver_name,pts_qualifying,pts_race,points_earned,starting_salary,salary_after_event,salary_change
0,MER,59,118,177.0,28.5,28.8,0.3
1,FER,51,110,161.0,22.5,23.7,1.2
2,RBS,45,82,127.0,15.0,17.4,2.4
3,HAS,37,88,125.0,5.0,8.0,3.0
4,MCL,51,72,123.0,28.5,26.3,-2.2
5,RBR,39,72,111.0,25.0,23.0,-2.0
6,ALP,32,76,108.0,12.5,12.9,0.4
7,AUD,41,62,103.0,10.0,10.4,0.4
8,WIL,26,70,96.0,17.5,15.5,-2.0
9,CAD,25,54,79.0,7.5,7.3,-0.2


## 3. Optimal Lineup

### Maximise points (unconstrained, £100M budget)

In [ ]:
lineup_pts = optimal_lineup(scored, optimize_for="points", budget=100)

display_cols = ["type", "driver_abbr", "driver_name", "starting_salary",
                "points_earned", "salary_change", "star"]

print(lineup_pts[display_cols].sort_values(["type", "points_earned"], ascending=[True, False]).to_string(index=False))

star_row = lineup_pts[lineup_pts["star"] == 1].iloc[0]
total_pts = lineup_pts["points_earned"].sum() + star_row["points_earned"]  # star counted twice
print(f"\nTotal salary used: £{lineup_pts['starting_salary'].sum():.1f}M")
print(f"Total points (starred driver doubled): {total_pts:.0f}")

  type driver_abbr  driver_name  starting_salary  points_earned  salary_change  star
driver         BEA   O. Bearman              9.2          172.0            2.0     1
driver         LIN  A. Lindblad              4.6          163.0            2.0     0
driver         ANT K. Antonelli             24.8          161.0            1.1     0
driver         HAM  L. Hamilton             20.9          156.0            1.2     0
driver         BOR G. Bortoleto             10.5          151.0            2.0     0
  team         MER          MER             28.5          177.0            0.3     0

Total salary used: £98.5M
Total points (starred driver doubled): 1152


### Maximise salary gain (to grow your budget for future races)

In [ ]:
lineup_sal = optimal_lineup(scored, optimize_for="salary_change", budget=100)

print(lineup_sal[display_cols].sort_values(["type", "salary_change"], ascending=[True, False]).to_string(index=False))
print(f"\nTotal salary used:   £{lineup_sal['starting_salary'].sum():.1f}M")
print(f"Total salary gained: £{lineup_sal['salary_change'].sum():.1f}M")

  type driver_abbr  driver_name  starting_salary  points_earned  salary_change  star
driver         LIN  A. Lindblad              4.6          163.0            2.0     0
driver         COL F. Colapinto              4.7          108.0            2.0     0
driver         OCO      E. Ocon              7.9          130.0            2.0     0
driver         BOR G. Bortoleto             10.5          151.0            2.0     0
driver         BEA   O. Bearman              9.2          172.0            2.0     0
  team         HAS          HAS              5.0          125.0            3.0     0

Total salary used:   £41.9M
Total salary gained: £13.0M


### With locked-in drivers

Suppose you already have HAM and LEC under contract (they cost nothing), and you have £60M left for the remaining 3 driver slots and 1 constructor slot.

In [ ]:
lineup_locked = optimal_lineup(
    scored,
    locked_in=["HAM", "LEC"],
    optimize_for="points",
    budget=60,
)

print(lineup_locked[display_cols].sort_values(["type", "points_earned"], ascending=[True, False]).to_string(index=False))

star_row = lineup_locked[lineup_locked["star"] == 1].iloc[0]
total_pts = lineup_locked["points_earned"].sum() + star_row["points_earned"]
free_picks = lineup_locked[~lineup_locked["driver_abbr"].isin(["HAM", "LEC"])]
print(f"\nBudget used for free picks: £{free_picks['starting_salary'].sum():.1f}M / £60M")
print(f"Total points (starred driver doubled): {total_pts:.0f}")

  type driver_abbr  driver_name  starting_salary  points_earned  salary_change  star
driver         BEA   O. Bearman              9.2          172.0            2.0     1
driver         LIN  A. Lindblad              4.6          163.0            2.0     0
driver         LEC   C. Leclerc             23.5          159.0            1.0     0
driver         HAM  L. Hamilton             20.9          156.0            1.2     0
driver         BOR G. Bortoleto             10.5          151.0            2.0     0
  team         MER          MER             28.5          177.0            0.3     0

Budget used for free picks: £52.8M / £60M
Total points (starred driver doubled): 1150


## 4. Score a Specific Team

Use `score_my_team` to evaluate a particular lineup you're considering — or check how your actual team did.

In [8]:
points, sal_change = score_my_team(
    australia,
    drivers=["RUS", "ANT", "LEC", "BEA", "LIN"],
    team="MER",
    star_driver="BEA",
    round=0,
)

total points: 1168
total salary change: 7.3
